In [1]:
"""
Evaluation Script with Rendering Window
========================================

Visualize a trained SAC agent performing the Lift task in Robosuite.

PREREQUISITES (Docker on Windows):
----------------------------------
1. Install VcXsrv on Windows: https://sourceforge.net/projects/vcxsrv/

2. Launch XLaunch with these settings:
   - Multiple windows
   - Start no client
   - CHECK "Disable access control" (important!)

3. Set DISPLAY in Docker (add to docker-compose.yml or export manually):
   export DISPLAY=host.docker.internal:0.0

4. Ensure the container was started AFTER setting DISPLAY


TROUBLESHOOTING:
----------------
- "Failed to open display": VcXsrv not running or DISPLAY not set
- "Could not initialize GLFW": Try `export MUJOCO_GL=glx` before running
- Black window: Check VcXsrv firewall permissions
"""

import os
# Override to use GLFW for window rendering (not EGL which is headless)
# GLX is shit!
os.environ["MUJOCO_GL"] = "glfw"
print(f"DISPLAY: {os.environ.get('DISPLAY', 'NOT SET')}")

DISPLAY: :0


In [2]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU Memory: 6.44 GB
GPU Memory Allocated: 0.00 GB


In [3]:
import time
from UpgradedEnvHRL import UpgradedEnvHRL
from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy


[robosuite WARNING] No private macro file found! (macros.py:57)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:58)
[robosuite WARNING] To setup, run: python /LearnFlake/src/external_pkgs/RoboSuite/robosuite/scripts/setup_macros.py (macros.py:59)
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
/usr/local/lib/python3.10/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pi

In [4]:

# =============================================================================
# CONFIGURATION - Modify these as needed
# =============================================================================
MODEL_PATH = "sac_rover2026_goal_hover.zip"
N_EPISODES = 5
ROBOT = "Rover2026"


In [5]:

# =============================================================================
# ENVIRONMENT SETUP
# =============================================================================
print("Creating environment with renderer enabled...")
env = UpgradedEnvHRL(
            render=True,
            domain_randomization=True,
            control_freq=20,
            horizon=150,
            controller="OSC_POSE",
            robot_name=ROBOT,
        )


[robosuite INFO] Loading controller configuration from: /LearnFlake/src/external_pkgs/RoboSuite/robosuite/controllers/config/robots/default_rover2026.json (composite_controller_factory.py:121)


Creating environment with renderer enabled...


In [ ]:

# =============================================================================
# LOAD MODEL
# =============================================================================
print(f"Loading model from: {MODEL_PATH}")

if not os.path.exists(MODEL_PATH):
    print(f"Model not found at {MODEL_PATH}. Train first!")
    env.close()
    exit(0)

model = SAC.load(MODEL_PATH, env=env, device="cuda")
print(f"Model trained for {model.num_timesteps} timesteps")


Loading model from: sac_rover2026_goal_hover.zip
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Model trained for 1000 timesteps


: 

In [ ]:

# =============================================================================
# RUN EVALUATION WITH RENDERING
# =============================================================================
print(f"\nRunning {N_EPISODES} evaluation episodes with rendering...")
print("=" * 50)

start_time = time.time()
mean_reward, std_reward = evaluate_policy(
    model,
    env,
    n_eval_episodes=N_EPISODES,
    render=False,
    deterministic=True
)
elapsed = time.time() - start_time

print("=" * 50)
print(f"Mean reward: {mean_reward:.2f}, Std reward: {std_reward:.2f}")
print(f"Evaluation took {elapsed:.2f} seconds")

env.close()
print("Evaluation complete!")



Running 5 evaluation episodes with rendering...


/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(
